In [ ]:
import pandas as pd


def merge_csvs(csv_list, output_path, source_str, replc=None, rename_patterns=None):
    """
    Merge multiple CSV files into one, report column mismatches,
    summarize missing values, and optionally rename column patterns.

    Parameters:
    -----------
    csv_list : list of str
        List of paths to CSV files.
    output_path : str
        Path to save the merged CSV.
    source_str : str or list
        Source label(s) to add.
    rename_patterns : list of tuple(str, str), optional
        List of (pattern, replacement) pairs to apply to column names.
        Example: [("af2", "alphafold2"), ("score_", "metric_")]
    """

    dfs = []
    all_columns = set()

    for i, csv_file in enumerate(csv_list):
        df = pd.read_csv(csv_file)

        # --- Optional column renaming ---
        if rename_patterns is not None:
            for src, repl in rename_patterns:
                df.columns = df.columns.str.replace(src, repl, regex=False)

        # Add source column
        if isinstance(source_str, list):
            df["source"] = source_str[i]

        dfs.append(df)
        all_columns.update(df.columns)

    # Check for column mismatches
    for i, df in enumerate(dfs):
        missing_in_df = all_columns - set(df.columns)
        extra_in_df = set(df.columns) - all_columns
        if missing_in_df or extra_in_df:
            print(f"Column mismatch in '{csv_list[i]}':")
            if missing_in_df:
                print(f"  Missing columns: {missing_in_df}")
            if extra_in_df:
                print(f"  Extra columns: {extra_in_df}")

    # Concatenate
    merged_df = pd.concat(dfs, ignore_index=True, sort=False)

    if not isinstance(source_str, list) and source_str is not None:
        merged_df["source"] = source_str

    # Save
    merged_df.to_csv(output_path, index=False)

    # Missing value summary
    missing_summary = merged_df.isna().sum()
    print("\nSummary of missing values:")
    for col, count in missing_summary.items():
        if count > 0:
            print(f"{col}: {count} missing")

    return merged_df

In [ ]:
df=pd.read_csv("data/prepared_training_dataset_publish.csv")
df.columns = df.columns.str.replace("dG_SASA_ratio", "dG_dSASA_ratio", regex=False)
df.columns = df.columns.str.replace("boltz1", "boltz", regex=False)

MODEL_ORDER = ["af2", "af3", "boltz", "colab", "input"]
MODEL_RANK = {m: i for i, m in enumerate(MODEL_ORDER)}

def canonicalize_rmsd_columns(df):
    new_cols = {}

    for col in df.columns:
        parts = col.split("_")

        # Only process RMSD columns
        if col.startswith("RMSD_") and len(parts) >= 4:
            model1 = parts[-2]
            model2 = parts[-1]

            if model1 in MODEL_RANK and model2 in MODEL_RANK:
                # sort according to predefined order
                m1, m2 = sorted([model1, model2], key=lambda x: MODEL_RANK[x])

                new_col = "_".join(parts[:-2] + [m1, m2])
                new_cols[col] = new_col

    return df.rename(columns=new_cols)

df=canonicalize_rmsd_columns(df)
df.to_csv("prepared_training_dataset_publish_column_names_updates.csv",index=False)




merge_csvs(csv_list=["prepared_training_dataset_publish_column_names_updates.csv",
                     "data/boltzgen_predicted_and_experimental_data.csv",
                     "data/bindcraft_predicted_and_experimental_data.csv"],
                     output_path="original_data_a_bindcraft_a_boltzgen_prepared_column_names_updated.csv",source_str=None,rename_patterns=[("boltz1","boltz"),("colabfold","colab")])
